# Уточняющие вопросы по тестовому кейсу

Ноутбук читает выбранное описание из `data/test` и строит два независимых списка из **7 вопросов в порядке важности**:

1. только по тексту кейса;
2. по тексту кейса с учетом каталога параметров из `output/cost_estimation_parameters.csv`.

Каталог используется как источник идей и примеров: модель должна игнорировать нерелевантные параметры. Первый запрос не получает каталог, поэтому списки можно корректно сравнить.

Перед запуском создайте `.env` с `OPENAI_API_KEY=...`. Модель задается через `OPENAI_MODEL`, а конкретный кейс — через `TEST_CASE_FILE` (имя файла или путь). Если `TEST_CASE_FILE` не задан, используется первый файл в `data/test`.

In [1]:
# При необходимости раскомментируйте:
# %pip install openai python-dotenv pandas

from pathlib import Path
import json
import os

import pandas as pd
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

## Настройки и входные данные

Чтобы выбрать кейс прямо в ноутбуке, замените `CASE_FILE` на имя файла из `data/test`, например `"case_082_optimal_drop_times_using_machine_learning.md"`. Абсолютный или существующий относительный путь также поддерживается.

In [2]:
TEST_DIR = Path("data/test")
PARAMETERS_CSV = Path("output/cost_estimation_parameters.csv")
CASE_FILE = os.getenv("TEST_CASE_FILE")  # Например: "case_082_optimal_drop_times_using_machine_learning.md"

load_dotenv()
MODEL = os.getenv("OPENAI_MODEL", "gpt-5-mini")
MAX_API_ATTEMPTS = 3
QUESTION_COUNT = 7

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("В .env не найден OPENAI_API_KEY")
if not TEST_DIR.is_dir():
    raise RuntimeError(f"Папка с тестовыми кейсами не найдена: {TEST_DIR}")
if not PARAMETERS_CSV.is_file():
    raise RuntimeError(f"Таблица параметров не найдена: {PARAMETERS_CSV}")

case_paths = sorted(path for path in TEST_DIR.iterdir() if path.is_file())
if not case_paths:
    raise RuntimeError(f"В {TEST_DIR} не найдено ни одного кейса")

if CASE_FILE:
    supplied_path = Path(CASE_FILE)
    case_path = supplied_path if supplied_path.is_file() else TEST_DIR / supplied_path
    if not case_path.is_file():
        available = ", ".join(path.name for path in case_paths)
        raise FileNotFoundError(f"Кейс не найден: {CASE_FILE}. Доступны: {available}")
else:
    case_path = case_paths[0]

case_text = case_path.read_text(encoding="utf-8").strip()
if not case_text:
    raise ValueError(f"Файл кейса пуст: {case_path}")

parameters_df = pd.read_csv(PARAMETERS_CSV, encoding="utf-8-sig").fillna("")
required_columns = {
    "параметр",
    "подробное описание параметра",
    "примеры значений",
    "важность (1-10)",
    "когда применять этот параметр оценки",
    "когда не применять этот параметр",
}
missing_columns = required_columns - set(parameters_df.columns)
if missing_columns:
    raise ValueError(f"В таблице отсутствуют столбцы: {sorted(missing_columns)}")

client = OpenAI()
print(f"Модель: {MODEL}")
print(f"Кейс: {case_path}")
print(f"Символов в описании: {len(case_text)}")
print(f"Параметров в каталоге: {len(parameters_df)}")

Модель: gpt-5-mini
Кейс: data/test/case_081_zillow_floor_plan_training_models_to_detect_windows_doors_and_openings_in_panora.md
Символов в описании: 20336
Параметров в каталоге: 71


## Промпты и проверка ответа

Оба режима требуют ровно 7 самостоятельных вопросов. Номер `1` означает самый важный вопрос. Проверка отклоняет ответы с неверным количеством, порядком, пустыми полями или дублями.

In [3]:
SYSTEM_PROMPT = """
Ты — ведущий системный аналитик, который готовит требования для самостоятельной разработки ML-, AI-, data- или backend-решения по описанию референсного кейса.
Сформулируй ровно 7 самых важных уточняющих вопросов, ответы на которые сильнее всего влияют на границы, архитектуру, трудоемкость и критерии приемки подобного решения.

Правила:
1. Пиши только на русском языке.
2. Расположи вопросы по убыванию важности: priority от 1 (самый важный) до 7.
3. Каждый пункт должен быть конкретным вопросом, на который заказчик может дать содержательный ответ.
4. Не спрашивай то, что уже однозначно указано в кейсе; неизвестные, предположительные и неоднозначные сведения нужно уточнять.
5. Не объединяй несколько независимых тем в один перегруженный вопрос.
6. Избегай дублей и общих вопросов без объяснимого влияния на разработку.
7. В why_important кратко объясни, какое решение или объем работ зависит от ответа.
8. Верни только корректный JSON без Markdown.

Формат: {"questions": [{"priority": 1, "question": "...", "why_important": "..."}]}
""".strip()


def normalize_questions(payload):
    questions = payload.get("questions")
    if not isinstance(questions, list) or len(questions) != QUESTION_COUNT:
        actual = len(questions) if isinstance(questions, list) else "не список"
        raise ValueError(f"Ожидалось {QUESTION_COUNT} вопросов, получено: {actual}")

    normalized = []
    for item in questions:
        if not isinstance(item, dict):
            raise ValueError("Каждый вопрос должен быть JSON-объектом")
        try:
            priority = int(item.get("priority"))
        except (TypeError, ValueError) as error:
            raise ValueError("priority должен быть целым числом") from error
        question = str(item.get("question", "")).strip()
        why_important = str(item.get("why_important", "")).strip()
        if not question or not why_important:
            raise ValueError("У вопроса пустое поле question или why_important")
        if not question.endswith("?"):
            raise ValueError(f"Формулировка должна быть вопросом: {question}")
        normalized.append({
            "priority": priority,
            "question": question,
            "why_important": why_important,
        })

    priorities = [item["priority"] for item in normalized]
    if priorities != list(range(1, QUESTION_COUNT + 1)):
        raise ValueError(f"Неверный порядок priority: {priorities}")
    canonical_questions = [item["question"].casefold() for item in normalized]
    if len(canonical_questions) != len(set(canonical_questions)):
        raise ValueError("В ответе есть одинаковые вопросы")
    return normalized


def request_questions(user_prompt):
    last_error = None
    for attempt in range(1, MAX_API_ATTEMPTS + 1):
        correction = "" if last_error is None else (
            f"\n\nПредыдущий ответ не прошел проверку: {last_error}. Исправь ответ."
        )
        response = client.responses.create(
            model=MODEL,
            instructions=SYSTEM_PROMPT,
            input=user_prompt + correction,
            text={"format": {"type": "json_object"}},
        )
        try:
            return normalize_questions(json.loads(response.output_text))
        except (json.JSONDecodeError, TypeError, ValueError) as error:
            last_error = str(error)
            print(f"Попытка {attempt} не прошла проверку: {last_error}")
    raise RuntimeError(f"Не удалось получить корректный список вопросов: {last_error}")

## 1. Вопросы только по описанию кейса

В этом запросе модель не видит таблицу параметров.

In [5]:
base_user_prompt = f"""
Подготовь уточняющие вопросы для самостоятельной разработки решения, подобного описанному ниже.

<case>
{case_text}
</case>

Верни результат строго в формате JSON.
""".strip()

questions_without_catalog = request_questions(base_user_prompt)

## 2. Вопросы с учетом каталога параметров

В запрос передается полный каталог. Инструкция разрешает использовать только релевантные строки и запрещает искусственно включать параметры ради покрытия таблицы.

In [7]:
catalog_records = parameters_df.to_dict(orient="records")
catalog_json = json.dumps(catalog_records, ensure_ascii=False)

catalog_user_prompt = f"""
Подготовь уточняющие вопросы для самостоятельной разработки решения, подобного описанному ниже.

Используй каталог параметров только как источник возможных идей, формулировок и примеров. Сначала оцени применимость каждого кандидата к конкретному кейсу. Не включай параметр только потому, что он присутствует в каталоге, и не пытайся обеспечить покрытие таблицы. Итог по-прежнему должен содержать только 7 наиболее важных вопросов для этого кейса.

<case>
{case_text}
</case>

<parameters_catalog>
{catalog_json}
</parameters_catalog>

Верни результат строго в формате JSON.
""".strip()

questions_with_catalog = request_questions(catalog_user_prompt)

## 3. Оба списка

In [8]:
def show_questions(title, questions):
    display(Markdown(f"### {title}"))
    frame = pd.DataFrame(questions).rename(columns={
        "priority": "Приоритет",
        "question": "Уточняющий вопрос",
        "why_important": "Почему это важно",
    })
    display(frame.style.hide(axis="index"))


show_questions("Без учета таблицы параметров", questions_without_catalog)
show_questions("С учетом таблицы параметров", questions_with_catalog)

### Без учета таблицы параметров

Приоритет,Уточняющий вопрос,Почему это важно
1,"Какие конкретные целевые метрики и пороги приёмки вы ожидаете для финального решения по каждой из трёх классов (например, AP@IoU=0.5 для дверей/окон/проёмов, минимальные требуемые precision/recall, допустимый разрыв по сравнению с human baseline)?","От ответа зависит выбор модели и архитектуры, объём и баланс датасета, стратегия валидации, а также критерии приёмки и объём доработок до продакшена."
2,"В каком формате downstream-пайплайн ожидает предсказания: достаточно ли пиксельных боксов в equirectangular-проекции, или нужен результат в координатах плана пола (например, X/Y в метрах) с учётом высоты/позы камеры или глубины?","Определяет необходимость добавления геометрических преобразований, модулей оценки позы/глубины, точность боксов по вертикали и интеграцию с генератором планов — существенно влияет на архитектуру и объём работ."
3,"Какие размеченные данные и метаданные вы готовы предоставить для разработки (точное число панорам с аннотациями по классам, формат аннотаций, примечания по спорным случаям, уровень меж-аннотататорного согласия)?","От наличия и качества аннотаций зависят сроки и трудозатраты на сбор/переразметку, необходимость увеличения датасета, а также выбор методов борьбы с дисбалансом и шума."
4,"Какие требования по latency/throughput и аппаратной платформе для инференса: максимально допустимое время обработки одной панорамы, целевая пропускная способность (панорам/сек или в сутки), и доступное оборудование на проде (GPU/CPU/edge)?","Ограничения по времени и железу определяют выбор между более точными тяжёлыми моделями и лёгкими/квантизированными решениями, необходимость батчинга, асинхронной обработки или предварительной обработки кадрами."
5,"Насколько входные панорамы стандартизованы: гарантирован ли 'leveled' equirectangular для всех снимков, фиксированное разрешение/аспект, и какие реальные вариации ожидаются (несоосность, разные разрешения, артефакты, зеркала, ночные/темные кадры)?","Если входы непостоянны, потребуется предобработка (выравнивание, нормализация разрешения, фильтрация), расширенная аугментация и более робастные модели — это увеличивает объём разработки и тестирования."
6,"Подтверждаете ли вы окончательную таксономию и правила включения для спорных случаев (например: считать ли душевые двери, шкафы, внешние двери, объекты в зеркале как класс door/opening/window)? Доступен ли финальный аннотаторский гайд или нужно его составлять/уточнять?","Единая финальная дефиниция классов критична для консистентности аннотаций, уменьшения ошибок модели и оценки — влияет на инструкции для разметки, ревью-процесс и постобработку предсказаний."
7,"Какой протокол оценки и тестовый бенчмарк вы хотите использовать в проекте и для CI: фиксированная тестовая выборка с N независимыми аннотаторами, какие IoU-thresholds (только 0.5 или набор как в COCO), и какие дополнительные метрики отслеживать (per-class AP, mAP, FP per pano, latency)?","От этого зависит реализация тестового хранилища, автоматизация валидации, сравнение с human baseline и критерии регресс-тестирования при последующих релизах."


### С учетом таблицы параметров

Приоритет,Уточняющий вопрос,Почему это важно
1,"Будут ли для каждого панорамного изображения доступны точные позы/координаты камеры (позиция x,y,z и ориентация), а также известна высота/уровень пола и калибровка камеры?","Наличие экструзий/поз камеры и уровня пола решает, требуется ли простая проекция боксов на план (простая геометрия) или нужно добавлять модуль оценки глубины/SLAM/позиционирования — это резко меняет архитектуру и вычислительные требования."
2,"Какой точный объём аннотированных панорам доступен сейчас для обучения/валидации/теста (числа для train/val/test), и планируется ли регулярное пополнение аннотаций/процесс QA (скорость пополнения, команда аннотаторов)?","Размер и долговременность датасета определяют выбор модели (fine-tune vs from-scratch), необходимость data augmentation, требуемое время тренировки и операционные процессы для поддержания качества."
3,"Какие целевые метрики и пороги приёмки вы ждёте от решения (например AP@IoU=0.5 для каждой из категорий: door/window/opening), и какие классы критичнее для downstream (приоритет классов)?","Конкретные целевые числа определяют объём доработок, подбор архитектуры, бюджет на сбор дополнительных аннотаций и критерии приёмочного тестирования и релиза."
4,"Где планируется запуск inference: облачная инфраструктура с GPU (укажите возможные типы GPU/число), CPU-only серверы или edge/он-премise устройства? Есть ли ограничения по памяти/времени выполнения на узел?","Окружение исполнения определяет допустимые модели (тяжёлые двухступенчатые сети или лёгкие one-stage/квантизированные), оптимизации (трассировка, TFLite/ONNX) и стоимость эксплуатации."
5,Каков ожидаемый объём обработки: среднее и пиковое число панорам/домов в день и требуемая параллельность (максимум одновременных панорам для обработки)?,"Throughput и пиковая нагрузка влияют на дизайн масштабируемости, очередей, batch-инференса, число инстансов и экономику решения."
6,"Какие требования по времени отклика на обработку одной панорамы: интерактивное (<2–30 секунд), near-real-time (несколько минут) или пакетная обработка (часы)?","SLA по задержкам определяет выбор между быстрыми лёгкими моделями/кроп-фьюжном и более точными, но медленными архитектурами; влияет на user experience и требования к параллелизации."
7,"Нужны ли от модели не только 2D боксы в изображении, но и абсолютные физические параметры для downstream генератора плана (ширина проёма в метрах, точная точка пересечения с полом, привязка к координатам плана)?","Требование выдавать реальные размеры/координаты потребует дополнительной информации (калибровка, глубина, позы) или построения отдельной геометрической/структурной подсистемы, существенно увеличивая объём работ."
